# Final Root Caries Risk Model

This notebook trains, evaluates, and saves the final logistic regression model for root caries risk prediction.

In [1]:

# Imports
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import joblib


## Feature Contract

In [2]:

FEATURE_COLUMNS = [
    "RIDAGEYR",
    "is_female",
    "n_missing_teeth",
    "n_filled_teeth",
    "ever_smoked",
    "INDFMPIR"
]

TARGET_COLUMN = "has_root_caries"


## Load Prepared Dataset

In [3]:

df = pd.read_parquet("../data/processed/final_caries_features.parquet")
df.head()


,RIDAGEYR,is_female,n_missing_teeth,n_filled_teeth,ever_smoked,INDFMPIR,has_root_caries
0,13.0,1,4,0,0,0.83,0
1,29.0,1,0,8,0,5.00,0
2,49.0,0,18,0,1,1.96,1
3,36.0,0,10,10,1,0.83,0
4,68.0,0,32,0,0,1.20,0


## Feature Preparation

In [4]:

def prepare_features(df):
    X = df[FEATURE_COLUMNS].copy()
    X["INDFMPIR"] = X["INDFMPIR"].fillna(X["INDFMPIR"].median())
    return X


## Train/Test Split and Model Training

In [5]:

X = prepare_features(df)
y = df[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=101, stratify=y
)

model = LogisticRegression(max_iter=1000, solver="lbfgs")
model.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Evaluation

In [6]:

y_prob = model.predict_proba(X_test)[:, 1]
threshold = 0.10
y_pred = (y_prob >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


[[1811  986]
 [  99  227]]
              precision    recall  f1-score   support

           0       0.95      0.65      0.77      2797
           1       0.19      0.70      0.29       326

    accuracy                           0.65      3123
   macro avg       0.57      0.67      0.53      3123
weighted avg       0.87      0.65      0.72      3123

ROC-AUC: 0.7332121839569565


## Odds Ratios

In [7]:

coef_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coef": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0])
}).sort_values("odds_ratio", ascending=False)

coef_df


,feature,coef,odds_ratio
4,ever_smoked,0.772053,2.164205
0,RIDAGEYR,0.028770,1.029188
2,n_missing_teeth,-0.011469,0.988596
3,n_filled_teeth,-0.048890,0.952286
1,is_female,-0.099081,0.905669
5,INDFMPIR,-0.352704,0.702785


## Save Model Artifacts

In [8]:
from pathlib import Path

Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(model, "../models/root_caries_logreg.joblib")

metadata = {
    "features": FEATURE_COLUMNS,
    "threshold": threshold,
    "model_type": "logistic_regression",
    "target": TARGET_COLUMN
}

joblib.dump(metadata, "../models/root_caries_metadata.joblib")


['../models/root_caries_metadata.joblib']

## Example Inference

In [9]:

example_patient = pd.DataFrame([{
    "RIDAGEYR": 65,
    "is_female": 0,
    "n_missing_teeth": 5,
    "n_filled_teeth": 8,
    "ever_smoked": 1,
    "INDFMPIR": 1.2
}])

risk = model.predict_proba(example_patient)[0, 1]
print("Predicted root caries risk:", risk)


Predicted root caries risk: 0.26254530761265404


In [10]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Compute ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

# Plot
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, linewidth=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Root Caries Risk Model")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

# Save
plt.savefig(
    "../reports/figures/roc_curve.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()

In [11]:
# Build odds ratio dataframe
coef_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "odds_ratio": np.exp(model.coef_[0])
}).sort_values("odds_ratio")

# Plot
plt.figure(figsize=(7, 5))
plt.barh(coef_df["feature"], coef_df["odds_ratio"])
plt.axvline(1.0, linestyle="--", linewidth=1)
plt.xlabel("Odds Ratio")
plt.title("Feature Effects on Root Caries Risk")
plt.grid(axis="x", alpha=0.3)

# Save
plt.savefig(
    "../reports/figures/odds_ratios.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()